In [264]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [265]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [266]:
apps_df = pd.read_csv("Play Store Data.csv")
reviews_df = pd.read_csv("User Reviews.csv")

In [267]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [268]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [269]:
#Step 2: Data Cleaning
apps_df = apps_df.dropna(subset = ['Rating'])
for column in apps_df.columns:
    apps_df[column].fillna(apps_df[column].mode()[0], inplace = True)
apps_df.drop_duplicates(inplace = True)

apps_df = apps_df = apps_df[apps_df['Rating'] <= 5]
reviews_df.dropna(subset = ['Translated_Review'], inplace = True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_1472\757697897.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  apps_df[column].fillna(apps_df[column].mode()[0], inplace = True)


In [270]:
apps_df.dtypes

App                   str
Category              str
Rating            float64
Reviews               str
Size                  str
Installs              str
Type                  str
Price                 str
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
dtype: object

In [271]:
#Convert the Installs columns to numeric by removing commas and +
apps_df['Installs'] = apps_df['Installs'].str.replace(',', '').str.replace('+', '').astype(int)

#Convert Price column to numeric after removing $
apps_df['Price'] = apps_df['Price'].str.replace('$', '').astype(float)

In [272]:
apps_df.dtypes

App                   str
Category              str
Rating            float64
Reviews               str
Size                  str
Installs            int64
Type                  str
Price             float64
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
dtype: object

In [273]:
reviews_df.dtypes

App                           str
Translated_Review             str
Sentiment                     str
Sentiment_Polarity        float64
Sentiment_Subjectivity    float64
dtype: object

In [274]:
merged_df = pd.merge(apps_df, reviews_df, on = 'App', how = 'inner')

In [275]:
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I hate,Negative,-0.800,0.900000


In [276]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M',''))
    elif 'k' in size:
        return float(size.replace('k',''))/1024
    else:
        return np.nan
apps_df['Size']=apps_df['Size'].apply(convert_size)

In [277]:
#Lograrithmic
apps_df['Log_Installs']=np.log(apps_df['Installs'])

In [278]:
apps_df['Reviews']=apps_df['Reviews'].astype(int)

In [279]:
apps_df['Log_Reviews']=np.log(apps_df['Reviews'])

In [280]:
apps_df.dtypes

App                   str
Category              str
Rating            float64
Reviews             int64
Size              float64
Installs            int64
Type                  str
Price             float64
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
Log_Installs      float64
Log_Reviews       float64
dtype: object

In [281]:
def rating_group(rating):
    if rating >= 4:
        return 'Top rated app'
    elif rating >= 3:
        return 'Above average'
    elif rating >= 2:
        return 'Average'
    else:
        return 'Below average'
apps_df['Rating_Group'] = apps_df['Rating'].apply(rating_group)

In [282]:
#Revenue column
apps_df['Revenue']=apps_df['Price']*apps_df['Installs']

In [283]:
sia = SentimentIntensityAnalyzer()
#Polarity Scores in SIA
#Positive, Negative, Neutral and Compound: -1 - Very negative ; +1 - Very positive

In [284]:
review = "This app is amazing! I love the new features."
sentiment_score= sia.polarity_scores(review)
print(sentiment_score)

{'neg': 0.0, 'neu': 0.42, 'pos': 0.58, 'compound': 0.8516}


In [285]:
reviews_df['Sentiment_Score']=reviews_df['Translated_Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

In [286]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Sentiment_Score
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,0.6597
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,0.6249
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,0.6369
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,0.6369


In [287]:
apps_df['Last Updated']=pd.to_datetime(apps_df['Last Updated'],errors='coerce')

In [288]:
apps_df['Year']=apps_df['Last Updated'].dt.year

In [289]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0,2018
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,13.122363,6.874198,Above average,0.0,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0,2018
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0,2018
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0,2018


In [290]:
html_files_path="./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [291]:
plot_containers=""

In [292]:
# Save each Plotly figure to an HTML file
def save_plot_as_html(fig, filename, insight):
    global plot_containers
    filepath = os.path.join(html_files_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    # Append the plot and its insight to plot_containers
    plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    fig.write_html(filepath, full_html=False, include_plotlyjs='inline')

In [293]:
plot_width=400
plot_height=300
plot_bg_color='black'
text_color='white'
title_font={'size':16}
axis_font={'size':12}

In [294]:
#Figure 1
category_counts=apps_df['Category'].value_counts().nlargest(10)
fig1=px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={'x':'Category','y':'Count'},
    title='Top Categories on Play Store',
    color=category_counts.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=400,
    height=300
)
fig1.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig1,"Category Graph 1.html","The top categories on the Play Store are dominated by tools, entertainment, and productivity apps")
            

In [295]:
#Figure 2
type_counts=apps_df['Type'].value_counts()
fig2=px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='App Type Distribution',
    color_discrete_sequence=px.colors.sequential.RdBu,
    width=400,
    height=300
)
fig2.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig2,"Type Graph 2.html","Most apps on the Playstore are free, indicating a strategy to attract users first and monetize through ads or in app purchases")

In [296]:
#Figure 3
fig3=px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#636EFA'],
    width=400,
    height=300
)
fig3.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig3,"Rating Graph 3.html","Ratings are skewed towards higher values, suggesting that most apps are rated favorably by users")

In [297]:
#Figure 4
sentiment_counts=reviews_df['Sentiment_Score'].value_counts()
fig4=px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    labels={'x':'Sentiment Score','y':'Count'},
    title='Sentiment Distribution',
    color=sentiment_counts.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=400,
    height=300
)
fig4.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)

save_plot_as_html(fig4,"Sentiment Graph 4.html","Sentiments in reviews show a mix of positive and negative feedback, with a slight lean towards positive sentiments")

In [298]:
#Figure 5
installs_by_category=apps_df.groupby('Category')['Installs'].sum().nlargest(10)
fig5=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    orientation='h',
    labels={'x':'Installs','y':'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Blues,
    width=400,
    height=300
)
fig5.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig5,"Installs Graph 5.html","The categories with the most installs are social and communication apps, reflecting their broad appeal and daily usage")

In [299]:
# Updates Per Year Plot
updates_per_year = apps_df['Last Updated'].dt.year.value_counts().sort_index()
fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x': 'Year', 'y': 'Number of Updates'},
    title='Number of Updates Over the Years',
    color_discrete_sequence=['#AB63FA'],
    width=plot_width,
    height=plot_height
)
fig6.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig6, "Updates Graph 6.html", "Updates have been increasing over the years, showing that developers are actively maintaining and improving their apps.")

In [300]:
#Figure 7
revenue_by_category=apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    labels={'x':'Category','y':'Revenue'},
    title='Revenue by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=400,
    height=300
)
fig7.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig7,"Revenue Graph 7.html","Categories such as Business and Productivity lead in revenue generation, indicating their monetization potential")

In [301]:
#Figure 8
genre_counts=apps_df['Genres'].str.split(';',expand=True).stack().value_counts().nlargest(10)
fig8=px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x':'Genre','y':'Count'},
    title='Top Genres',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.OrRd,
    width=400,
    height=300
)
fig8.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig8,"Genre Graph 8.html","Action and Casual genres are the most common, reflecting users' preference for engaging and easy-to-play games")

In [302]:
#Figure 9
fig9=px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=400,
    height=300
)
fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig9,"Update Graph 9.html","The Scatter Plot shows a weak correlation between the last update and ratings, suggesting that more frequent updates dont always result in better ratings.")

In [303]:
#Figure 10
fig10=px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=400,
    height=300
)
fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig10,"Paid Free Graph 10.html","Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for")

In [304]:
#Figure 11
#Task 1: Use a grouped bar chart to compare the average rating and total review count for the top 10 app categories by number of installs.
#Filter out any categories where the average rating is below 4.0 and size below 10 M and last update should be Jan month . 
#This graph should work only between 3PM IST to 5 PM IST apart from that time we should not show this graph in dashboard itself.


from datetime import datetime
from zoneinfo import ZoneInfo

current_time = datetime.now(ZoneInfo("Asia/Kolkata")).time()

start_time = datetime.strptime("15:00", "%H:%M").time()
end_time = datetime.strptime("17:00", "%H:%M").time()

if start_time <= current_time < end_time:

    # Filter data
    filtered_df = apps_df[(apps_df['Size'] >= 10) & (apps_df['Last Updated'].dt.month == 1)].copy()

    # Category-level calculations
    category_summary = (filtered_df.groupby('Category')
        .agg(
            Average_Rating=('Rating', 'mean'),
            Total_Reviews=('Reviews', 'sum'),
            Total_Installs=('Installs', 'sum')
        ).reset_index()
    )

    # Remove categories with average rating below 4
    category_summary = category_summary[category_summary['Average_Rating'] >= 4.0]

    # Top 10 categories by installs
    category_summary = category_summary.sort_values('Total_Installs', ascending=False).head(10)

    # Reshape data
    chart_df = category_summary.melt(
        id_vars=['Category'],
        value_vars=['Average_Rating', 'Total_Reviews'],
        var_name='Metric',
        value_name='Value'
    )

    # Grouped bar chart
    fig11 = px.bar(
        chart_df,
        x='Category',
        y='Value',
        color='Metric',
        barmode='group',
        labels={'Category': 'App Category', 'Value': 'Value','Metric': ''},
        title='Top 10 App Categories by Installs',
        width=400,
        height=300
    )

    # Styling
    fig11.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size': 16},
        xaxis=dict(title='App Category',title_font={'size': 12},tickangle=-45),
        yaxis=dict(title='Value',title_font={'size': 12}),
        margin=dict(l=10,r=10,t=30,b=10)
    )
    save_plot_as_html(fig11,"Top 10 App Categories Graph 11.html","Family apps have the highest total review count, followed by Games and Sports, indicating stronger user engagement in these categories.")

else:
    print("This graph is available only between 3 PM and 5 PM IST.")

    

This graph is available only between 3 PM and 5 PM IST.


In [305]:
#Figure 12
#Task 2- Create an interactive Choropleth map using Plotly to visualize global installs by Category. 
#Apply filters to show data for only the top 5 app categories and highlight category where the number of installs exceeds 1 million. 
#The app category should not start with the characters “A,” “C,” “G,” or “S.” 
#This graph should work only between 6 PM IST and 8 PM IST; apart from that time, we should not show it in the dashboard itself.


current_time = datetime.now(ZoneInfo("Asia/Kolkata")).time()

start_time = datetime.strptime("18:00", "%H:%M").time()
end_time = datetime.strptime("20:00", "%H:%M").time()

if start_time <= current_time < end_time:

    #Filter Categories
    filtered_df = apps_df[~apps_df['Category'].str.upper().str.startswith(('A', 'C', 'G', 'S'),na=False)].copy()
    
    #Grouping Category by Total Installs
    category_installs = filtered_df.groupby('Category')['Installs'].sum().reset_index()

    category_installs = (category_installs.sort_values('Installs', ascending = False).head(5))

    #Identifying categories exceeding 1 million installs
    category_installs['Install_Level'] = (
        category_installs['Installs']
        .apply(
            lambda x:'Above 1 Million'
            if x > 1_000_000
            else '1 Million or Less')
    )

    #Creating interactive horizontal bar chart
    fig12 = px.bar(
        category_installs,
        x='Installs',
        y='Category',
        orientation='h',
        color='Install_Level',
        hover_data={'Category': True,'Installs': ':,', 'Install_Level': True},
        labels={'Installs': 'Total Installs','Category': 'App Category','Install_Level': 'Install Level'},
        title='Top 5 App Categories by Total Installs',
        color_discrete_map={'Above 1 Million': '#FF7043','1 Million or Less': '#5DADE2'},
        width=400,
        height=300
    )

    #Adding 1 Million reference line
    fig12.add_vline(
        x=1000000,
        line_dash='dash',
        line_width=2,
        annotation_text='1 Million Installs',
        annotation_position='top'
    )

    fig12.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size': 16},
        xaxis=dict(title='Total Installs',title_font={'size': 12},tickformat=','),
        yaxis=dict(title='App Category',title_font={'size': 12}),
        margin=dict(l=10,r=10,t=30,b=10),
        legend_title_text=''
    )
    save_plot_as_html(fig12,"Top 5 Categoreis by Installs Graph 12.html","Productivity has the highest total installs among the selected top 5 categories, while all displayed categories exceed the 1 million install threshold, indicating strong user adoption across these categories.")

else:
    print("This graph is available only between 6 PM and 8 PM IST.")



This graph is available only between 6 PM and 8 PM IST.


In [306]:
# Figure 13
#Task 3- Create a dual-axis chart comparing the average installs and revenue for free vs. paid apps within the top 3 app categories. 
#Apply filters to exclude apps with fewer than 10,000 installs and revenue below $10,000 and android version should be more than 4.0 
#as well as size should be more than 15M and content rating should be Everyone and app name should not have more than 30 characters 
#including space and special character .this graph should work only between 1 PM IST to 2 PM IST apart from that time 
#we should not show this graph in dashboard itself.


import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

current_time = datetime.now(ZoneInfo("Asia/Kolkata")).time()
start_time = datetime.strptime("13:00", "%H:%M").time()
end_time = datetime.strptime("14:00", "%H:%M").time()

if start_time <= current_time < end_time:

    df = apps_df.copy()

    df['App_Name_Length'] = (df['App'].astype(str).str.len())
    df['Android_Version_Num'] = (df['Android Ver'].astype(str).str.extract(r'(\d+(?:\.\d+)?)')[0].astype(float))

    filtered_df = df[
        (df['Installs'] >= 10000) &
        (df['Revenue'] >= 10000) &
        (df['Android_Version_Num'] > 4.0) &
        (df['Size'] > 15) &
        (df['Content Rating'] == 'Everyone') &
        (df['App_Name_Length'] <= 30)
    ].copy()

    # Find top 3 categories by total installs

    top_3_categories = ( filtered_df.groupby('Category')['Installs'].sum().nlargest(3).index)

    top_3_df = filtered_df[filtered_df['Category'].isin(top_3_categories)].copy()

    # Calculate average installs and revenue

    summary = (top_3_df.groupby(['Category', 'Type']).agg(Average_Installs=('Installs', 'mean'),
            Average_Revenue=('Revenue', 'mean')).reset_index())

    fig13 = make_subplots(specs=[[{"secondary_y": True}]])

    installs_fig = px.bar(
        summary,
        x='Category',
        y='Average_Installs',
        color='Type',
        barmode='group',
        color_discrete_sequence=px.colors.qualitative.Pastel
    )

    for trace in installs_fig.data:

        fig13.add_trace(
            go.Bar(
                x=trace.x,
                y=trace.y,
                name=f'{trace.name} - Avg Installs',
                offsetgroup=trace.name,
                legendgroup=trace.name,
                hovertemplate=(
                    '<b>%{x}</b><br>'
                    f'Type: {trace.name}<br>'
                    'Average Installs: %{y:,.0f}'
                    '<extra></extra>'
                )
            ),
            secondary_y=False
        )

    revenue_fig = px.bar(
        summary,
        x='Category',
        y='Average_Revenue',
        color='Type',
        barmode='group',
        color_discrete_sequence=px.colors.qualitative.Pastel
    )

    for trace in revenue_fig.data:

        fig13.add_trace(
            go.Bar(
                x=trace.x,
                y=trace.y,
                name=f'{trace.name} - Avg Revenue',
                offsetgroup=trace.name,
                legendgroup=trace.name,
                hovertemplate=(
                    '<b>%{x}</b><br>'
                    f'Type: {trace.name}<br>'
                    'Average Revenue: $%{y:,.2f}'
                    '<extra></extra>'
                ),
                showlegend=False
            ),
            secondary_y=True
        )

    fig13.update_layout(
        barmode='group',
        title='Average Installs and Revenue: Free vs Paid Apps',
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size': 16},
        width=400,
        height=300,
        margin=dict(l=10,r=10,t=30,b=10),
        legend=dict(orientation='h',yanchor='bottom',y=1.02,xanchor='center',x=0.5),
        xaxis=dict(title='App Category',title_font={'size': 12})) 

    fig13.update_yaxes(
        title_text='Average Installs',
        title_font={'size': 12},
        secondary_y=False
    )

    fig13.update_yaxes(
        title_text='Average Revenue ($)',
        title_font={'size': 12},
        secondary_y=True
    )

    save_plot_as_html(fig13,"Avg Installs and Revenue Graph 13.html","Photography generates the highest average revenue among the selected categories, while Personalization has the highest average installs for paid apps, highlighting differences between user reach and monetization")

else:
    print("This graph is available only between 1 PM and 2 PM IST.")
    

This graph is available only between 1 PM and 2 PM IST.


In [307]:
#Figure 14
#Task 4- Plot a time series line chart to show the trend of total installs over time, segmented by app category. 
#Highlight periods of significant growth by shading the areas under the curve where the increase in installs exceeds 20% month-over-month 
#and app name should not starts with x, y ,z and app category should start with letter " E " or " C " or " B " and
#We have to translate the Beauty category in Hindi and Business category in Tamil and Dating category in German while showing it on Graph. 
#reviews should be more than 500 the app name should not contain letter "S" as well as 
#this graph should work only between 6 PM IST to 9 PM IST apart from that time we should not show this graph in dashboard itself


current_time = datetime.now(ZoneInfo("Asia/Kolkata")).time()
start_time = datetime.strptime("18:00", "%H:%M").time()
end_time = datetime.strptime("21:00", "%H:%M").time()

if start_time <= current_time < end_time:
    df = apps_df.copy()

    df = df[~df['App'].astype(str).str.startswith(('X', 'Y', 'Z'),na=False)]
    df = df[~df['App'].astype(str).str.contains('S',case=False, na=False)]
    df = df[df['Category'].astype(str).str.startswith(('B', 'C', 'E'),na=False)]
    df = df[df['Reviews'] > 500]

    #Create monthly time period
    df['Month'] = (pd.to_datetime(df['Last Updated']).dt.to_period('M').dt.to_timestamp())

    #Calculate total installs by month and category
    monthly_installs = (df.groupby(['Month', 'Category'])['Installs'].sum().reset_index())

    #Calculate month-over-month growth
    monthly_installs['MoM_Growth'] = (monthly_installs.groupby('Category')['Installs'].pct_change())

    category_translation = {
        'BEAUTY': 'सुंदरता',       #Hindi
        'BUSINESS': 'வணிகம்',      #Tamil
        'DATING': 'Partnersuche'   #German
    }

    monthly_installs['Category_Display'] = (monthly_installs['Category'].map(category_translation).fillna(monthly_installs['Category']))

    #Create line chart
    fig14 = px.line(
        monthly_installs,
        x='Month',
        y='Installs',
        color='Category_Display',
        markers=True,
        labels={'Month': 'Month','Installs': 'Total Installs', 'Category_Display': 'App Category'},
        title='Monthly Install Trend by App Category',
        color_discrete_sequence=px.colors.qualitative.Pastel,
        width=400,
        height=300
    )

    #Highlight months with more than 20% growth
    high_growth_months = monthly_installs[monthly_installs['MoM_Growth'] > 0.20]['Month'].unique()

    for month in high_growth_months:
        next_month = month + pd.offsets.MonthBegin(1)

        fig14.add_vrect(
            x0=month,
            x1=next_month,
            fillcolor='rgba(255, 255, 255, 0.18)',
            line_width=0,
            layer='above',
            annotation_text='>20% MoM',
            annotation_position='top left',
            annotation_font=dict(color='white',size=9)
        )

    fig14.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size': 16},
        xaxis=dict(title='Month',title_font={'size': 12}),
        yaxis=dict(title='Total Installs',title_font={'size': 12}),
        legend=dict(title='App Category'),
        margin=dict(l=10, r=10, t=30, b=10)
    )

    save_plot_as_html(fig14,"Monthly Install Trend by App Category Graph 14.html","Communication shows the most significant growth in installs, with a sharp increase around 2018, while the other selected categories remain comparatively low throughout the period")

else:
    print("This graph is available only between 6 PM and 9 PM IST.")

This graph is available only between 6 PM and 9 PM IST.


In [308]:
#Figure 15
#Task 5- Plot a bubble chart to analyze the relationship between app size (in MB) and average rating, with the bubble size 
#representing the number of installs. Include a filter to show only apps with a rating higher than 3.5 and 
#that belong to the Game, Beauty ,business , commics , commication , Dating , Entertainment , social and event categories. 
#Reviews should be greater than 500 and the app name should not contain letter "S" and sentiment subjectivity should be more than 0.5 
#and highlight the Game Category chart in Pink color. We have to translate the Beauty category in Hindi and Business category in Tamil
#and Dating category in German while showing it on Graphs. Installs should be more than 50k as well as 
#this graph should work only between 5 PM IST to 7 PM IST apart from that time we should not show this graph in dashboard itself

current_time = datetime.now(ZoneInfo("Asia/Kolkata")).time()

start_time = datetime.strptime("17:00", "%H:%M").time()
end_time = datetime.strptime("19:00", "%H:%M").time()

if start_time <= current_time < end_time:
    
    df = merged_df.copy()
    df['Reviews'] = (df['Reviews'].str.replace(',', '', regex=False).str.replace('+', '', regex=False))
    df['Reviews'] = pd.to_numeric(df['Reviews'],errors='coerce')

    df['Size'] = (df['Size'].str.replace('M', '', regex=False).str.replace('m', '', regex=False).str.strip())
    df['Size'] = pd.to_numeric(df['Size'],errors='coerce')

    app_df = (df.groupby(['App','Category','Rating','Reviews','Size','Installs'],as_index=False)
        .agg(Sentiment_Subjectivity=('Sentiment_Subjectivity','mean')))

    required_categories = ['GAME','BEAUTY','BUSINESS','COMICS','COMMUNICATION','DATING','ENTERTAINMENT','SOCIAL','EVENTS']

    filtered_df = app_df[
        (app_df['Rating'] > 3.5) &
        (app_df['Category'].str.upper().isin(required_categories)) &
        (app_df['Reviews'] > 500) &
        (~app_df['App'].str.contains('S',case=False,na=False)) &
        (app_df['Sentiment_Subjectivity'] > 0.5) &
        (app_df['Installs'] > 50000)
    ].copy()

    filtered_df = filtered_df.dropna(subset=['Size','Rating','Installs','Sentiment_Subjectivity'])

    category_translation = {
        'BEAUTY': 'सुंदरता',
        'BUSINESS': 'வணிகம்',
        'DATING': 'Partnersuche'
    }

    filtered_df['Category_Display'] = filtered_df['Category'].str.upper().map(category_translation).fillna(filtered_df['Category'].str.title())

    fig15 = px.scatter(
        filtered_df,
        x='Size',
        y='Rating',
        size='Installs',
        color='Category_Display',
        hover_name='App',
        hover_data={
            'Size': ':.2f',
            'Rating': ':.2f',
            'Installs': ':,',
            'Reviews': ':,',
            'Sentiment_Subjectivity': ':.2f'
        },
        labels={
            'Size': 'App Size (MB)',
            'Rating': 'Average Rating',
            'Installs': 'Number of Installs',
            'Category_Display': 'App Category',
            'Sentiment_Subjectivity':
                'Sentiment Subjectivity'
        },
        title='App Size vs Average Rating — Bubble Size = Installs',
        size_max=45,
        width=400,
        height=300
    )

    category_colors = {
        'Game': '#FF1493',
        'सुंदरता': '#F5B041',
        'வணிகம்': '#5DADE2',
        'Partnersuche': '#58D68D',
        'Comics': '#AF7AC5',
        'Communication': '#48C9B0',
        'Entertainment': '#F4D03F',
        'Social': '#E67E22',
        'Events': '#3498DB'
    }

    for trace in fig15.data:
        if trace.name in category_colors:trace.marker.color = (category_colors[trace.name])

    fig15.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size': 16},
        xaxis=dict(title='App Size (MB)',title_font={'size': 12}),
        yaxis=dict(title='Average Rating',title_font={'size': 12}),
        legend=dict(title='App Category'),
        margin=dict(l=10,r=10,t=30,b=10))

    save_plot_as_html(fig15,"App Size vs Average Rating Graph 15.html","Apps with higher ratings are distributed across a wide range of sizes, while the largest install volumes are concentrated around apps rated approximately 4.2 to 4.5, suggesting no clear relationship between app size and rating")

else:

    print("This graph is available only between 5 PM and 7 PM IST.")
    

This graph is available only between 5 PM and 7 PM IST.


In [309]:
#Figure 16
#Task 6- You are required to create a stacked area chart to visualize the cumulative number of installs over time for each app category, 
#with each category represented as a separate color band in the chart. Apply the following filters before plotting: 
#include only apps with an average rating of at least 4.2, app names that do not contain any numbers, 
#app categories that start with the letter “T” or “P,” reviews greater than 1,000, and app sizes between 20 MB and 80 MB. 
#In the chart legend, translate “Travel & Local” into French, “Productivity” into Spanish, and “Photography” into Japanese. 
#Highlight by increasing the color intensity for any month where total installs increased by more than 25% month-over-month for any category. 
#This visualization must only be displayed between 4 PM IST and 6 PM IST, and it should not appear on the dashboard outside this time window.


current_time = datetime.now(ZoneInfo("Asia/Kolkata")).time()
start_time = datetime.strptime("16:00", "%H:%M").time()
end_time = datetime.strptime("18:00", "%H:%M").time()

if start_time <= current_time < end_time:

    df = apps_df.copy()
    
    filtered_df = df[
        (df['Rating'] >= 4.2) &
        (~df['App'].str.contains(r'\d', regex=True, na=False)) &
        (df['Category'].str.upper().str.startswith(('T', 'P'), na=False)) &
        (df['Reviews'] > 1000) &
        (df['Size'] >= 20) & (df['Size'] <= 80)
    ].copy()

    filtered_df = filtered_df.dropna(subset=['Last Updated'])

    filtered_df['Month'] = (filtered_df['Last Updated'].dt.to_period('M').dt.to_timestamp())

    monthly_installs = (filtered_df.groupby(['Month', 'Category'], as_index=False)['Installs'].sum())

    all_months = pd.date_range(start=monthly_installs['Month'].min(),end=monthly_installs['Month'].max(),freq='MS')

    all_categories = sorted(monthly_installs['Category'].unique())
    
    complete_index = pd.MultiIndex.from_product([all_months, all_categories],names=['Month', 'Category'])

    monthly_installs = (monthly_installs.set_index(['Month', 'Category']).reindex(complete_index, fill_value=0).reset_index())

    monthly_installs['MoM_Growth'] = (monthly_installs.groupby('Category')['Installs'].pct_change())

    growth_by_month = (monthly_installs.groupby('Month')['MoM_Growth'].max().reset_index())

    growth_by_month['High_Growth'] = (growth_by_month['MoM_Growth'] > 0.25)

    high_growth_months = (growth_by_month[growth_by_month['High_Growth']]['Month'].tolist())

    monthly_installs['Cumulative_Installs'] = (monthly_installs.sort_values(['Category', 'Month']).groupby('Category')['Installs'].cumsum())

    category_translation = {
        'TRAVEL & LOCAL': 'Voyage et local',   # French
        'TRAVEL_AND_LOCAL': 'Voyage et local', # French
        'PRODUCTIVITY': 'Productividad',       # Spanish
        'PHOTOGRAPHY': '写真'                  # Japanese
    }

    monthly_installs['Category_Display'] = (monthly_installs['Category'].str.upper().map(category_translation).fillna(
        monthly_installs['Category'].str.title()))

    fig16 = px.area(
        monthly_installs,
        x='Month',
        y='Cumulative_Installs',
        color='Category_Display',
        title='Cumulative Installs Over Time by App Category',
        color_discrete_sequence=px.colors.qualitative.Pastel,
        width=400,
        height=300
    )

    # Hover format
    fig16.update_traces(
        hovertemplate=(
            '<b>%{fullData.name}</b><br>'
            'Month: %{x|%b %Y}<br>'
            'Cumulative Installs: %{y:,}'
            '<extra></extra>'
        )
    )

    # Highlight months with >25% MoM growth
    for month in high_growth_months:

        # Find next month to determine highlight width
        next_month = (month + pd.offsets.MonthBegin(1))

        fig16.add_vrect(
            x0=month,
            x1=next_month,
            fillcolor='rgba(255, 255, 255, 0.18)',
            line_width=0,
            layer='above'
        )

    fig16.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title=('Cumulative Installs Over Time by App Category'),
        title_font={'size': 16},
        xaxis=dict(title='Month',title_font={'size': 12},showgrid=False),
        yaxis=dict( title='Cumulative Installs',title_font={ 'size': 12},tickformat=','),
        legend=dict(title='App Category'),
        hovermode='x unified',
        margin=dict(l=10,r=10,t=30,b=10),
    )
    fig16.show()
    save_plot_as_html(fig16,"Cumalative Installs by Category Graph 16.html","App installs remain relatively low in the earlier years before rising sharply around 2018, with Photography contributing the largest cumulative install volume among the selected categories")

else:
    print("This graph is available only between 4 PM and 6 PM IST.")

This graph is available only between 4 PM and 6 PM IST.


In [310]:
plot_containers_split=plot_containers.split('</div>')

In [311]:
if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [312]:
dashboard_html= """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name=viewport" content="width=device-width,initial-scale-1.0">
    <title> Google Play Store Review Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify_content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container: hover .insights {{
            display: block;
        }}
        </style>
        <script>
            function openPlot(filename) {{
                window.open(filename, '_blank');
                }}
        </script>
    </head>
    <body>
        <div class= "header">
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/4a/Logo_2013_Google.png/800px-Logo_2013_Google.png" alt="Google Logo">
            <h1>Google Play Store Reviews Analytics</h1>
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/78/Google_Play_Store_badge_EN.svg/1024px-Google_Play_Store_badge_EN.svg.png" alt="Google Play Store Logo">
        </div>
        <div class="container">
            {plots}
        </div>
    </body>
    </html>
    """

In [313]:
final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [314]:
dashboard_path=os.path.join(html_files_path,"web page.html")

In [315]:
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)


In [316]:
webbrowser.open('file://'+os.path.realpath(dashboard_path))

True